# 08 — Preparação para Streamlit

Neste notebook, vamos organizar os arquivos finais do projeto para a aplicação em Streamlit.

A modelagem já foi concluída. Agora o foco é transformar a análise em entrega.

Vamos preparar:

- série histórica;
- métricas dos modelos;
- previsões no período de teste;
- resumo do modelo final;
- arquivos padronizados para o app.

### 08.01 Imports e caminhos

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

import plotly.graph_objects as go

if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import (
    CORES,
    aplicar_layout_padrao,
    grafico_linha_padrao
)

caminho_outputs_tabelas = raiz_projeto / "outputs" / "tabelas"

caminho_serie_historica = caminho_outputs_tabelas / "serie_historica.csv"
caminho_metricas = caminho_outputs_tabelas / "metricas.csv"
caminho_previsoes_teste = caminho_outputs_tabelas / "previsoes_teste.csv"

print("Arquivos finais esperados:")
print("-", caminho_serie_historica)
print("-", caminho_metricas)
print("-", caminho_previsoes_teste)

## Validando os arquivos finais

Antes de criar o app, precisamos garantir que os arquivos principais existem e podem ser carregados.

Essa etapa evita erros no Streamlit.

### 08.02 Verificar existência dos arquivos

In [ ]:
arquivos_esperados = {
    "serie_historica": caminho_serie_historica,
    "metricas": caminho_metricas,
    "previsoes_teste": caminho_previsoes_teste
}

for nome, caminho in arquivos_esperados.items():
    if caminho.exists():
        print(f"OK - {nome}: {caminho}")
    else:
        print(f"ERRO - arquivo não encontrado: {caminho}")

### 08.03 Carregar arquivos finais

In [ ]:
serie_historica = pd.read_csv(caminho_serie_historica)
metricas = pd.read_csv(caminho_metricas)
previsoes_teste = pd.read_csv(caminho_previsoes_teste)

serie_historica["data_hora"] = pd.to_datetime(serie_historica["data_hora"])
previsoes_teste["data_hora"] = pd.to_datetime(previsoes_teste["data_hora"])

print("Série histórica:", serie_historica.shape)
print("Métricas:", metricas.shape)
print("Previsões no teste:", previsoes_teste.shape)

### 08.04 Conferir série histórica

In [ ]:
serie_historica.head()

### 08.05 Conferir métricas

In [ ]:
metricas

### 08.06 Conferir previsões

In [ ]:
previsoes_teste.head()

## Modelo final

Vamos identificar automaticamente o modelo com menor MAPE.

Esse será o modelo destacado na aplicação.

### 08.07 Identificar melhor modelo

In [ ]:
metricas_ordenadas = metricas.sort_values("MAPE").reset_index(drop=True)

melhor_modelo = metricas_ordenadas.iloc[0]

nome_melhor_modelo = melhor_modelo["modelo"]
mae_melhor_modelo = melhor_modelo["MAE"]
rmse_melhor_modelo = melhor_modelo["RMSE"]
mape_melhor_modelo = melhor_modelo["MAPE"]

print("Melhor modelo:")
print(nome_melhor_modelo)

print("\nMétricas:")
print(f"MAE: {mae_melhor_modelo:.2f}")
print(f"RMSE: {rmse_melhor_modelo:.2f}")
print(f"MAPE: {mape_melhor_modelo:.2f}%")

### 08.08 Criar resumo do projeto

In [ ]:
resumo_projeto = pd.DataFrame({
    "item": [
        "modelo_final",
        "mae",
        "rmse",
        "mape",
        "inicio_serie",
        "fim_serie",
        "quantidade_dias_historico",
        "quantidade_dias_teste"
    ],
    "valor": [
        nome_melhor_modelo,
        round(mae_melhor_modelo, 2),
        round(rmse_melhor_modelo, 2),
        round(mape_melhor_modelo, 2),
        serie_historica["data_hora"].min(),
        serie_historica["data_hora"].max(),
        serie_historica.shape[0],
        previsoes_teste.shape[0]
    ]
})

resumo_projeto

## Visualização final da série histórica

Este é o gráfico que pode ser usado no app para apresentar o comportamento geral da demanda.

### 08.09 Gráfico da série histórica

In [ ]:
fig = grafico_linha_padrao(
    df=serie_historica,
    x="data_hora",
    y="demanda",
    titulo="Série histórica da demanda de bicicletas",
    labels={
        "data_hora": "Data",
        "demanda": "Demanda diária"
    },
    altura=500
)

fig.show()

## Visualização final das previsões

Agora vamos conferir o gráfico que compara o valor real com os modelos no período de teste.

### 08.10 Gráfico final de comparação

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["demanda_real"],
        mode="lines",
        name="Real",
        line=dict(color="#0B1F4D", width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_baseline"],
        mode="lines",
        name="Baseline",
        line=dict(color="#7A7A7A", width=2, dash="dash")
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_arima"],
        mode="lines",
        name="ARIMA",
        line=dict(color="#6FA8FF", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_sarima"],
        mode="lines",
        name="SARIMA",
        line=dict(color="#F28E2B", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_sarimax"],
        mode="lines",
        name="SARIMAX",
        line=dict(color="#2CA02C", width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Comparação das previsões no período de teste",
    altura=500
)

fig.show()

## Padronizando arquivos para o app

Para simplificar o Streamlit, vamos salvar os arquivos finais com nomes diretos:

- `serie_historica.csv`
- `metricas.csv`
- `previsoes.csv`
- `resumo_projeto.csv`

Todos ficarão dentro de `outputs/tabelas`.

### 08.11 Salvar arquivos padronizados

In [ ]:
caminho_previsoes_app = caminho_outputs_tabelas / "previsoes.csv"
caminho_resumo_projeto = caminho_outputs_tabelas / "resumo_projeto.csv"

previsoes_teste.to_csv(
    caminho_previsoes_app,
    index=False,
    encoding="utf-8-sig"
)

resumo_projeto.to_csv(
    caminho_resumo_projeto,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos padronizados salvos:")
print("-", caminho_serie_historica)
print("-", caminho_metricas)
print("-", caminho_previsoes_app)
print("-", caminho_resumo_projeto)

### 08.12 Validar arquivos do app

In [ ]:
arquivos_app = [
    caminho_serie_historica,
    caminho_metricas,
    caminho_previsoes_app,
    caminho_resumo_projeto
]

for caminho in arquivos_app:
    df_teste = pd.read_csv(caminho)
    print(f"{caminho.name}: {df_teste.shape[0]} linhas e {df_teste.shape[1]} colunas")

## Próximo passo

Os arquivos finais estão organizados para o Streamlit.

Na próxima etapa, vamos construir o `app.py` para apresentar:

- a série histórica;
- a comparação dos modelos;
- as previsões no período de teste;
- o resumo do modelo final.